In [ ]:
import os
import numpy as np
import pandas as pd
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

print("1. Ham Veri (Bronze) Okunuyor ve Hazırlanıyor...")
current_dir = os.getcwd()
bronze_path = os.path.abspath(os.path.join(current_dir, "..", "data", "bronze"))
dt = DeltaTable(bronze_path)
df = dt.to_pandas()

# Önceki adımdaki zenginleştirmeyi (çeşitlendirmeyi) kalıcı olması için tekrar uyguluyoruz
df['timestamp'] = pd.to_datetime(df['timestamp'])
rastgele_dakikalar = np.random.randint(0, 24*60*30, size=len(df)) # Son 1 aya dağıtıyoruz
df['timestamp'] = df['timestamp'] - pd.to_timedelta(rastgele_dakikalar, unit='m')
olay_tipleri = ['Enerji Tüketimi', 'Sistem Uyarısı', 'Cihaz Arızası', 'Aşırı Yüklenme', 'Normal Kapanış']
df['event_type'] = np.random.choice(olay_tipleri, size=len(df))
df['user_id'] = np.random.randint(1, 51, size=len(df))

print("-" * 40)
print("2. ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING) BAŞLIYOR...")

# --- ÖZELLİK 1: Hafta Sonu Kontrolü ---
# İş Mantığı: Sistem hataları ve tüketim alışkanlıkları hafta içi ve hafta sonu farklı dinamiklere sahiptir.
df['is_weekend'] = df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
print("✅ Özellik 1 Üretildi: is_weekend (Hafta sonu ise 1, hafta içi ise 0)")

# --- ÖZELLİK 2: Günün Vakti (Time of Day) ---
# İş Mantığı: Olayların sabah, öğle, akşam veya gece yaşanması, anormallik tespiti için kritik bir bağlamdır.
def get_time_of_day(hour):
    if 6 <= hour < 12: return 'Sabah'
    elif 12 <= hour < 18: return 'Öğle'
    elif 18 <= hour < 24: return 'Akşam'
    else: return 'Gece'
df['time_of_day'] = df['timestamp'].dt.hour.apply(get_time_of_day)
print("✅ Özellik 2 Üretildi: time_of_day (Sabah, Öğle, Akşam, Gece)")

# --- ÖZELLİK 3: Kullanıcı İşlem Hacmi (User Event Frequency) ---
# İş Mantığı: Bir kullanıcının sistemde çok fazla işlem yapması "Power User" olduğunu veya bir bot/saldırı ihtimalini gösterir.
user_freq = df.groupby('user_id').size().to_dict()
df['user_event_count'] = df['user_id'].map(user_freq)
print("✅ Özellik 3 Üretildi: user_event_count (Kullanıcının toplam işlem sayısı)")

# --- ÖZELLİK 4: Öğe Popülerliği (Item Popularity) ---
# İş Mantığı: İlgili nesnenin (related_id) sistemde ne kadar sık kullanıldığı, sistem yükü dağılımı için önemlidir.
item_freq = df.groupby('related_id').size().to_dict()
df['item_popularity'] = df['related_id'].map(item_freq)
print("✅ Özellik 4 Üretildi: item_popularity (İlgili öğenin toplam etkileşim sayısı)")

# --- ÖZELLİK 5: Olay Risk Skoru (Event Risk Score) ---
# İş Mantığı: Her olayın iş etkisine göre risk ağırlığı vardır. Arızalar ve uyarılar modele daha yüksek ağırlıkla beslenmelidir.
risk_weights = {
    'Cihaz Arızası': 5, 
    'Aşırı Yüklenme': 4, 
    'Sistem Uyarısı': 3, 
    'Enerji Tüketimi': 2, 
    'Normal Kapanış': 1
}
df['event_risk_score'] = df['event_type'].map(risk_weights)
print("✅ Özellik 5 Üretildi: event_risk_score (Olayın kritiklik seviyesine göre 1-5 arası puan)")

print("-" * 40)
print("3. İŞ MANTIĞI ÖZETİ :")
print("""
Oluşturulan 5 özelliğin temel amacı, ham veri içindeki gizli örüntüleri algoritmaların anlayabileceği 
matematiksel ve bağlamsal formatlara çevirmektir. Zaman boyutundan (Hafta sonu/Günün vakti), kullanıcı 
davranışından (İşlem Hacmi) ve olayların kritiklik seviyelerinden (Risk Skoru) faydalanılarak 
kapsamlı bir model besleme tablosu oluşturulmuştur.
""")

print("-" * 40)
print("4. Delta Lake'e Kaydediliyor (Silver Layer)...")
# Özelliklerin eklendiği yeni tabloyu 'silver' veya 'features' katmanı olarak kaydediyoruz
silver_path = os.path.abspath(os.path.join(current_dir, "..", "data", "silver_features"))

# Spark'a hiç bulaşmadan, doğrudan deltalake kütüphanesiyle veriyi yazıyoruz
write_deltalake(silver_path, df, mode="overwrite")

print(f"🎉 BAŞARILI! Özellik tabloları Delta formatında şu yola kaydedildi: \n{silver_path}")

print("\nYeni Tablonun İlk 3 Satırı:")
display(df[['timestamp', 'is_weekend', 'time_of_day', 'user_event_count', 'item_popularity', 'event_risk_score']].head(3))

1. Ham Veri (Bronze) Okunuyor ve Hazırlanıyor...
----------------------------------------
2. ÖZELLİK MÜHENDİSLİĞİ (FEATURE ENGINEERING) BAŞLIYOR...
✅ Özellik 1 Üretildi: is_weekend (Hafta sonu ise 1, hafta içi ise 0)
✅ Özellik 2 Üretildi: time_of_day (Sabah, Öğle, Akşam, Gece)
✅ Özellik 3 Üretildi: user_event_count (Kullanıcının toplam işlem sayısı)
✅ Özellik 4 Üretildi: item_popularity (İlgili öğenin toplam etkileşim sayısı)
✅ Özellik 5 Üretildi: event_risk_score (Olayın kritiklik seviyesine göre 1-5 arası puan)
----------------------------------------
3. İŞ MANTIĞI ÖZETİ (Hoca/Değerlendirici için):

Oluşturulan 5 özelliğin temel amacı, ham veri içindeki gizli örüntüleri algoritmaların anlayabileceği 
matematiksel ve bağlamsal formatlara çevirmektir. Zaman boyutundan (Hafta sonu/Günün vakti), kullanıcı 
davranışından (İşlem Hacmi) ve olayların kritiklik seviyelerinden (Risk Skoru) faydalanılarak 
kapsamlı bir model besleme tablosu oluşturulmuştur.

------------------------------------

,timestamp,is_weekend,time_of_day,user_event_count,item_popularity,event_risk_score
0,2026-04-29 03:18:18.251362+00:00,0,Gece,348,1,1
1,2026-04-19 14:22:18.262107+00:00,1,Öğle,345,1,1
2,2026-05-09 00:21:18.272849+00:00,1,Gece,347,1,3
